# 🌾 FOODGUARD AI: Closed-Loop Food Resilience Operating System
### El Niño Scenario: Tamil Nadu -> Rice -> Next 30 Days

This Colab notebook contains the core Machine Learning engine for **FORESIGHT**. It follows the **Detect → Predict → Explain → Decide → Act → Verify → Learn** loop.

**Instructions for Teammates:**
1. Upload your real-world CSV data in the `DATA LOADING` section.
2. The model uses XGBoost to predict *Shortage Probability* and *Price Increase*.
3. It uses SHAP to provide *Root-Cause Explainability* (why the crisis is happening).
4. It includes an *Intervention Simulator* to show the impact of our recommended actions.

In [ ]:
# 📦 STEP 0: INSTALL DEPENDENCIES
!pip install xgboost shap pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

shap.initjs()
plt.style.use('dark_background')

## 🔍 1. DETECT & LOAD REAL-WORLD DATA
> *Teammate: Replace the mock generation below with your real dataset using `pd.read_csv('your_real_data.csv')`*

In [ ]:
# ---------------------------------------------------------
# TODO FOR TEAMMATE: LOAD YOUR REAL DATA HERE
# df = pd.read_csv('tamil_nadu_rice_data.csv')
# ---------------------------------------------------------

# For the hackathon demo, if the real data is still being cleaned, 
# here is a realistic synthetic data generator based on the El Niño Tamil Nadu scenario.
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    'rainfall_deficit_pct': np.random.uniform(0, 40, n_samples),      # 0 to 40% deficit
    'temp_anomaly_c': np.random.uniform(0, 3.5, n_samples),           # 0 to 3.5 C increase
    'reservoir_level_pct': np.random.uniform(20, 90, n_samples),      # 20% to 90% full
    'transport_disruption_pct': np.random.uniform(0, 30, n_samples),  # Logistics delays
    'current_stock_tonnes': np.random.uniform(10000, 50000, n_samples), 
    'projected_demand_tonnes': np.random.uniform(20000, 60000, n_samples)
})

# Create the Target Variables (What we want to predict)
# 1. Production Drop (Impacted by weather & water)
df['production_drop_pct'] = (df['rainfall_deficit_pct'] * 0.4) + (df['temp_anomaly_c'] * 2.5) - (df['reservoir_level_pct'] * 0.1)
df['production_drop_pct'] = np.clip(df['production_drop_pct'], 0, 100)

# 2. Shortage Probability (0 to 1) -> Impacted by drop, stock, and transport
shortage_risk = (df['projected_demand_tonnes'] - (df['current_stock_tonnes'] * (1 - df['production_drop_pct']/100))) / df['projected_demand_tonnes']
shortage_risk = shortage_risk + (df['transport_disruption_pct'] / 100)
df['shortage_probability'] = np.clip(shortage_risk, 0, 1)

# 3. Price Increase (Impacted by shortage probability)
df['price_increase_pct'] = df['shortage_probability'] * 45.0 + np.random.normal(0, 2, n_samples) # Up to 45% price spike

df.head()

## 🔮 2. PREDICT: XGBoost Risk Engine
Train the model to predict `Shortage Probability` and `Price Increase`.

In [ ]:
features = ['rainfall_deficit_pct', 'temp_anomaly_c', 'reservoir_level_pct', 
            'transport_disruption_pct', 'current_stock_tonnes', 'projected_demand_tonnes']

X = df[features]
y_shortage = df['shortage_probability']
y_price = df['price_increase_pct']

X_train, X_test, ys_train, ys_test, yp_train, yp_test = train_test_split(X, y_shortage, y_price, test_size=0.2, random_state=42)

# Train Shortage Model
model_shortage = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=4, learning_rate=0.1)
model_shortage.fit(X_train, ys_train)

# Train Price Model
model_price = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=4, learning_rate=0.1)
model_price.fit(X_train, yp_train)

print(f"Shortage Model RMSE: {mean_squared_error(ys_test, model_shortage.predict(X_test), squared=False):.4f}")
print(f"Price Model RMSE: {mean_squared_error(yp_test, model_price.predict(X_test), squared=False):.4f}")

## 🧠 3. EXPLAIN: Root-Cause Intelligence (SHAP)
Why is a specific district at risk? Judges love this. It makes the AI a glass box, not a black box.

In [ ]:
# Pick a specific "At Risk" district scenario (e.g., Chengalpattu under El Nino)
sample_district = X_test.iloc[[5]] # Select one district from test set

explainer = shap.TreeExplainer(model_shortage)
shap_values = explainer.shap_values(sample_district)

print("========== ROOT CAUSE ANALYSIS FOR TARGET DISTRICT ==========")
print(f"Predicted Shortage Probability: {model_shortage.predict(sample_district)[0]*100:.1f}%")
print("\nDRIVERS OF CRISIS:")
shap.force_plot(explainer.expected_value, shap_values[0], sample_district.iloc[0], matplotlib=True)

## ⚖️ 4. DECIDE & ACT: Optimization Engine
Don't just predict the crisis. Reduce it. This runs the `Buffer Optimizer` & `Transport Rerouting`.

In [ ]:
def ai_intervention_engine(district_data):
    """Simulates the AI deciding how to intervene."""
    data = district_data.copy()
    original_risk = model_shortage.predict(data)[0]
    original_price = model_price.predict(data)[0]
    
    interventions_taken = []
    
    # Tactic 1: Smart Inventory / Buffer Injection
    if original_risk > 0.30:
        injected_tonnes = data['projected_demand_tonnes'].values[0] * 0.15 # Inject 15% of demand
        data['current_stock_tonnes'] += injected_tonnes
        interventions_taken.append(f"Moved {int(injected_tonnes)} tonnes from surplus buffer.")
        
    # Tactic 2: Transport Rerouting (Bypass bottlenecks)
    if data['transport_disruption_pct'].values[0] > 10:
        data['transport_disruption_pct'] = max(0, data['transport_disruption_pct'].values[0] - 15) # Reroute reduces delay
        interventions_taken.append("Rerouted logistics via Option A (Avoids NH-XXX bottleneck).")
        
    new_risk = model_shortage.predict(data)[0]
    new_price = model_price.predict(data)[0]
    
    return original_risk, new_risk, original_price, new_price, interventions_taken

orig_risk, new_risk, orig_price, new_price, actions = ai_intervention_engine(sample_district)

## 🎯 5. VERIFY: Impact Monitor Dashboard
This is the **WOW Moment**. Show the judges the *Before vs. After*.

In [ ]:
print("┌─────────────────────────────────────┐")
print("│        INTERVENTION RESULT          │")
print("├─────────────────────────────────────┤")
for action in actions:
    print(f"│ Action: {action[:26]}...")
print("├─────────────────────────────────────┤")
print(f"│ Shortage risk       {orig_risk*100:.1f}% → {new_risk*100:.1f}%      │")
print(f"│ Price pressure      +{orig_price:.1f}% → +{new_price:.1f}%     │")
people_protected = int((orig_risk - new_risk) * 2100000) # Assuming 2.1M baseline
print(f"│ People protected    {people_protected:,}           │")
print("└─────────────────────────────────────┘")